In [30]:
import plotly.graph_objects as go
import numpy as np

class ArbitraryWaveformGenerator:
    c = 3e8
    def __init__(self, pulse_type="rectangle", sequence=[], length=100e-12, low=0.0, high=1.0,
                 sampling_rate=1e18):
        if pulse_type not in ["rectangle", "smooth_sin"]:
            raise ValueError("pulse_type должен быть 'rectangle' или 'smooth_sin'")
        self.pulse_type = pulse_type
        self.sequence = sequence
        self.length = length
        self.sampling_rate = sampling_rate
        self.low = low
        self.high = high

    def read_sequence_from_file(self, filename):
        with open(filename, 'r') as f:
            data = f.read().strip()
        self.sequence = [int(bit) for bit in data if bit in "01"]

    def get_sequence(self):
        return self.sequence

    def set_sequence(self, seq):
        if all(bit in [0, 1] for bit in seq):
            self.sequence = seq
        else:
            raise ValueError("Последовательность должна состоять из 0 и 1")

    def generate_pulses(self, smooth_factor=0.5):
        if not self.sequence:
            raise ValueError("Последовательность пуста")
        pulses = []
        slot_samples = int(self.length * self.sampling_rate)
        t = np.linspace(0, np.pi, slot_samples, endpoint=False)  # половина периода
        for bit in self.sequence:
            if self.pulse_type == "rectangle":
                pulse = bit * np.ones(slot_samples)
            elif self.pulse_type == "smooth_sin":
                # сигнал ±1, центрирован на 0, smooth_factor регулирует амплитуду сглаживания
                envelope = smooth_factor * np.sin(t)
                pulse = bit * envelope
            pulses.append(pulse)
        self._signal = np.concatenate(pulses)
        self._time = np.linspace(0, self.length * len(self.sequence),
                                 slot_samples * len(self.sequence), endpoint=False)
        return self._signal

    def plot_sequence(self):
        if not hasattr(self, "_signal"):
            raise ValueError("Сначала нужно сгенерировать импульсы методом generate_pulses")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=self._time, y=self._signal, mode='lines', name='Pulse'))
        fig.update_layout(
            width=800,
            height=600,
            margin=dict(l=50, r=10, t=10, b=50),
            xaxis_title="Время",
            yaxis_title="Амплитуда",
            xaxis=dict(tickfont=dict(size=12)),
            yaxis=dict(tickfont=dict(size=12)),
            template="plotly_white",
        )
        fig.show()

In [20]:
import numpy as np
import plotly.graph_objects as go

class ArbitraryWaveformGenerator:
    def __init__(self, mode="rec", sampling_rate=1e18, length=1,
                 low=0.0, high=1.0, duty=0.5, amp=1.0, phi=0.0, per=1e-12,
                 sequence=None, **kwargs):
        self.mode = mode
        self.sampling_rate = sampling_rate
        self.length = length
        self.low = low
        self.high = high
        self.duty = duty
        self.amp = amp
        self.phi = phi
        self.per = per
        self.sequence = np.array(sequence) if sequence is not None else np.array([])

        self.t = None
        self.signal = None

    def generate(self):
        # создаём временную ось
        self.t = np.arange(0, self.length, 1/self.sampling_rate)
        self.signal = np.zeros_like(self.t)

        if self.mode == "rec":
            # квадратный сигнал: level = high в течение duty*per, else low
            phase = (self.t + self.phi/self.per) % self.per
            self.signal = np.where(phase < self.duty*self.per, self.high, self.low)

        elif self.mode == "sin":
            # синус: amp * sin(2*pi*t/per + phi)
            self.signal = self.amp * np.sin(2*np.pi*self.t/self.per + self.phi)

        elif self.mode == "saw":
            # пилообразный сигнал: линейно от low к high на duty*per, затем сброс
            phase = (self.t + self.phi/self.per) % self.per
            self.signal = self.low + (self.high - self.low) * (phase / (self.duty*self.per))
            # ограничиваем сверху high
            self.signal = np.minimum(self.signal, self.high)

        elif self.mode == "seq":
            pass  # пока оставляем пустым
        else:
            raise ValueError("mode должен быть 'rec', 'sin', 'saw' или 'seq'")

        return self.signal

IndentationError: unindent does not match any outer indentation level (<string>, line 21)

In [1]:
import plotly.graph_objects as go
import numpy as np

class Encoder:
    low = 0
    high = 1
    def __init__(self, pulse_type="rectangle", sequence=[], length=100e-12):
        if pulse_type not in ["rectangle", "smooth_sin"]:
            raise ValueError("pulse_type должен быть 'rectangle' или 'smooth_sin'")
        self.pulse_type = pulse_type
        self.sequence = sequence
        self.length = length
        self.sampling_rate = 1e12

    def read_sequence_from_file(self, filename):
        with open(filename, 'r') as f:
            data = f.read().strip()
        self.sequence = [int(bit) for bit in data if bit in "01"]

    def get_sequence(self):
        return self.sequence

    def set_sequence(self, seq):
        if all(bit in [0, 1] for bit in seq):
            self.sequence = seq
        else:
            raise ValueError("Последовательность должна состоять из 0 и 1")

    def generate_pulses(self):
        if not self.sequence:
            raise ValueError("Последовательность пуста")

        slot_samples = int(self.length * self.sampling_rate)
        pulses = []

        for bit in self.sequence:
            pulse_level = self.high if bit == 1 else self.low
            pulse = np.ones(slot_samples) * pulse_level
            pulses.append(pulse)

        self._signal = np.concatenate(pulses)
        self._time = np.linspace(0, self.length * len(self.sequence),
                                 slot_samples * len(self.sequence), endpoint=False)
        return self._signal

    def plot_sequence(self):
        if not hasattr(self, "_signal"):
            raise ValueError("Сначала нужно сгенерировать импульсы методом generate_pulses")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=self._time, y=self._signal, mode='lines', name='Pulse'))
        fig.update_layout(
            width=800,
            height=600,
            margin=dict(l=50, r=10, t=10, b=50),
            xaxis_title="Время",
            yaxis_title="Амплитуда",
            xaxis=dict(tickfont=dict(size=12)),
            yaxis=dict(tickfont=dict(size=12)),
            template="plotly_white",
        )
        fig.show()

In [18]:
import random

enc = Encoder(pulse_type="rectangle", length=10e-12)  # длина слота под один бит
sequence_length = 100
enc.set_sequence([random.randint(0,1) for _ in range(sequence_length)])
enc.sampling_rate = 1e12
enc.generate_pulses()
enc.plot_sequence()

In [ ]:
enc = ArbitraryWaveformGenerator(pulse_type="smooth_sin")
enc.set_sequence([0,0,1,1,0,0])
enc.generate_pulses(smooth_factor=0.1)
enc.plot_sequence()